In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, io, data, features, viz, models
from fraud_detect.models import ModelBackend, ModelResult, CrossValidationResult

warnings.filterwarnings("ignore")
viz.configure_style()

## 10. Advanced Model Training

Train LightGBM, XGBoost, and CatBoost with cross-validation.
Compare feature importance across backends. This establishes
the advanced-model baseline before hyperparameter tuning.

> Training logic in `fraud_detect.models`.
> Base params in `fraud_detect.config`.

In [ ]:
train = io.load_train_features()
print(f"Data loaded: {train.shape}")
print(f"Fraud rate : {train['isFraud'].mean()*100:.2f}%")

### 10.1 LightGBM

In [ ]:
# 5-fold CV
cv_lgb = models.cross_validate_model(
    train, backend=ModelBackend.LIGHTGBM, cv_folds=5,
)
print(f"Mean AUC: {cv_lgb.mean_auc:.5f} ± {cv_lgb.std_auc:.5f}")
print(f"Fold scores: {[round(s,5) for s in cv_lgb.scores]}")

In [ ]:
# Full model with early stopping
result_lgb = models.train_model(train, ModelBackend.LIGHTGBM)
print(f"Train AUC: {result_lgb.train_auc:.5f}")
print(f"Val AUC  : {result_lgb.val_auc:.5f}")
print(f"Gap      : {result_lgb.train_auc - result_lgb.val_auc:.5f}")
print(f"Time     : {result_lgb.training_time:.1f}s")

In [ ]:
# Top-20 feature importance by gain
top20_lgb = result_lgb.feature_importance.head(20) if result_lgb.feature_importance is not None else pd.DataFrame()
if not top20_lgb.empty:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top20_lgb["feature"].iloc[::-1], top20_lgb["importance"].iloc[::-1], color="#4c72b0")
    ax.set_xlabel("Gain importance")
    ax.set_title("LightGBM — Top 20 features by gain")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "lgbm_feature_importance_top20.png")
    plt.show()
else:
    print("No feature importance available.")

### 10.2 XGBoost

In [ ]:
cv_xgb = models.cross_validate_model(
    train, backend=ModelBackend.XGBOOST, cv_folds=5,
)
print(f"Mean AUC: {cv_xgb.mean_auc:.5f} ± {cv_xgb.std_auc:.5f}")
print(f"Fold scores: {[round(s,5) for s in cv_xgb.scores]}")

In [ ]:
result_xgb = models.train_model(train, ModelBackend.XGBOOST)
print(f"Train AUC: {result_xgb.train_auc:.5f}")
print(f"Val AUC  : {result_xgb.val_auc:.5f}")
print(f"Gap      : {result_xgb.train_auc - result_xgb.val_auc:.5f}")
print(f"Time     : {result_xgb.training_time:.1f}s")

In [ ]:
if result_xgb.feature_importance is not None and not result_xgb.feature_importance.empty:
    top20_xgb = result_xgb.feature_importance.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top20_xgb["feature"].iloc[::-1], top20_xgb["importance"].iloc[::-1], color="#dd8452")
    ax.set_xlabel("Gain importance")
    ax.set_title("XGBoost — Top 20 features by gain")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "xgb_feature_importance_top20.png")
    plt.show()

### 10.3 CatBoost

In [ ]:
cv_cb = models.cross_validate_model(
    train, backend=ModelBackend.CATBOOST, cv_folds=5,
)
print(f"Mean AUC: {cv_cb.mean_auc:.5f} ± {cv_cb.std_auc:.5f}")
print(f"Fold scores: {[round(s,5) for s in cv_cb.scores]}")

In [ ]:
result_cb = models.train_model(train, ModelBackend.CATBOOST)
print(f"Train AUC: {result_cb.train_auc:.5f}")
print(f"Val AUC  : {result_cb.val_auc:.5f}")
print(f"Gap      : {result_cb.train_auc - result_cb.val_auc:.5f}")
print(f"Time     : {result_cb.training_time:.1f}s")

In [ ]:
if result_cb.feature_importance is not None:
    top20_cb = result_cb.feature_importance.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top20_cb["feature"].iloc[::-1], top20_cb["importance"].iloc[::-1], color="#55a868")
    ax.set_xlabel("PredictionValues importance")
    ax.set_title("CatBoost — Top 20 features")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "cb_feature_importance_top20.png")
    plt.show()

### 10.4 Cross-Validation Comparison

In [ ]:
cv_results = {
    "LightGBM": cv_lgb,
    "XGBoost": cv_xgb,
    "CatBoost": cv_cb,
}

cv_comparison = pd.DataFrame([{
    "Model": name, "CV Mean AUC": f"{cv.mean_auc:.5f}",
    "CV Std AUC": f"{cv.std_auc:.5f}", "CV Time (s)": f"{cv.training_time:.1f}",
} for name, cv in cv_results.items()])
cv_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(cv_results))
means = [cv.mean_auc for cv in cv_results.values()]
stds = [cv.std_auc for cv in cv_results.values()]
labels = list(cv_results.keys())

bars = ax.bar(x_pos, means, yerr=stds, capsize=6, color=["#4c72b0", "#dd8452", "#55a868"])
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("ROC-AUC")
ax.set_title("5-Fold CV — Mean AUC with ±1 Std")
ax.set_ylim(0.85, 1.0)

for bar, mean in zip(bars, means, strict=True):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{mean:.4f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
viz.save_figure(fig, config.METADATA_DIR / "cv_comparison.png")
plt.show()

### 10.5 Full Training Comparison

In [ ]:
full_results = {
    "LightGBM": result_lgb, "XGBoost": result_xgb, "CatBoost": result_cb,
}
full_comparison = pd.DataFrame([{
    "Model": name, "Train AUC": f"{r.train_auc:.5f}",
    "Val AUC": f"{r.val_auc:.5f}",
    "Overfitting Gap": f"{r.train_auc - r.val_auc:.5f}",
    "Time (s)": f"{r.training_time:.1f}",
} for name, r in full_results.items()])
full_comparison

In [ ]:
# Train vs validation AUC
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(full_results))
train_aucs = [r.train_auc for r in full_results.values()]
val_aucs = [r.val_auc for r in full_results.values()]
labels = list(full_results.keys())

width = 0.35
ax.bar(x_pos - width/2, train_aucs, width, label="Train AUC", color="#4c72b0", alpha=0.8)
ax.bar(x_pos + width/2, val_aucs, width, label="Val AUC", color="#c44e52", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("ROC-AUC")
ax.set_title("Train vs Validation AUC per Model")
ax.legend()
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
viz.save_figure(fig, config.METADATA_DIR / "train_vs_val_comparison.png")
plt.show()

### 10.6 Summary

| Model | CV Mean AUC | Val AUC | Overfitting Gap | Training Time |
|---|---|---|---|---|

Next: notebook 11 — hyperparameter tuning with Optuna.